In [1]:
"""
Isolate Analysis Notebook

This notebook compares allele profiles between metagenomic samples (actual) and 
cultured isolates from the same individuals. It analyzes:
1. Site overlap: Which genomic positions are present in both datasets
2. Allele concordance: Whether major alleles match between actual and isolate samples
3. Significance tracking: How well isolates preserve statistically significant sites
   identified in the metagenomic data

The analysis focuses on a specific MAG (Metagenome-Assembled Genome) and evaluates
how well cultured isolates represent the in vivo population genetics.
"""

import pandas as pd
import os
import gzip
import numpy as np
import logging
from tqdm import tqdm
from functools import partial
from multiprocessing import Pool, cpu_count

# Configure logging to provide informative output
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    datefmt='%Y-%m-%d %H:%M:%S')

In [2]:
# Core Functions for Profile Analysis

def read_profile(filepath):
    """
    Reads a gzipped allele profile file.
    
    Profile files contain position-by-position allele counts (A, C, G, T) for 
    a genomic region. These are generated by allele profiling tools.
    
    Args:
        filepath: Path to the gzipped TSV profile file
        
    Returns:
        DataFrame with columns: contig, position, A, C, G, T (and other metadata)
        Returns None if file doesn't exist or is missing required columns
    """
    if not isinstance(filepath, str) or not os.path.exists(filepath):
        logging.info(f"Profile file not found or path is invalid: {filepath}")
        return None

    profile_df = pd.read_csv(filepath, sep="\t", compression="gzip", dtype={'contig': str, 'position': 'int64'})
    
    # Ensure all required columns are present for analysis
    required_cols = ['contig', 'position', 'A', 'C', 'G', 'T']
    if not all(col in profile_df.columns for col in required_cols):
        logging.warning(f"File {filepath} is missing required columns. Skipping.")
        return None
    return profile_df


def get_major_alleles(profile_df):
    """
    Identifies the major (most abundant) allele(s) at each genomic position.
    
    This vectorized function efficiently finds which allele(s) have the maximum
    count at each position. Multiple alleles can tie for the maximum.
    
    Args:
        profile_df: DataFrame with allele count columns ['A', 'C', 'G', 'T']
        
    Returns:
        pandas Series where each value is a list of major alleles (e.g., ['A'], ['C', 'T'])
        Empty list returned for positions with no coverage
    """
    allele_cols = ['A', 'C', 'G', 'T']
    
    # Extract counts array, filling NaNs for robustness
    counts = profile_df[allele_cols].fillna(0).to_numpy(dtype=int)
    
    # Determine max count per row
    max_counts = counts.max(axis=1)
    
    # Identify rows with any coverage (max > 0)
    has_coverage = max_counts > 0
    
    # Build boolean mask where count == max_count for each row
    # [:, None] broadcasts max_counts to match the shape of counts
    # Only consider rows that have coverage
    mask = (counts == max_counts[:, None]) & (has_coverage[:, None])
    
    # Construct list of alleles per row using the boolean mask
    alleles = np.array(allele_cols)
    major_alleles_list = [alleles[row_mask].tolist() for row_mask in mask]
    
    return pd.Series(major_alleles_list, index=profile_df.index)


def check_allele_match(row):
    """
    Determines if the major alleles match between actual and isolate samples.
    
    A match occurs when at least one major allele is shared between the two samples.
    For example, if actual has ['A', 'C'] and isolate has ['C', 'T'], this counts 
    as a match because 'C' is shared.
    
    Args:
        row: DataFrame row with columns 'present', 'major_act', 'major_iso'
        
    Returns:
        True if there's any overlap between major alleles, False otherwise
    """
    # A site must be present in the isolate to be considered for a match
    if not row['present']:
        return False
        
    act_alleles = row['major_act']  # Major alleles from actual (metagenomic) sample
    iso_alleles = row['major_iso']  # Major alleles from isolate sample
    
    # Ensure both are non-empty lists before checking for intersection
    if not act_alleles or not iso_alleles:
        return False
        
    # Return True if there's any overlap between the two lists of major alleles
    return len(set(act_alleles).intersection(set(iso_alleles))) > 0


def compare_profiles(row, significant_p_sites=None, significant_q_sites=None):
    """
    Compares allele profiles between an actual (metagenomic) sample and its 
    corresponding isolate culture.
    
    This is the main comparison function designed for parallel execution. It:
    1. Loads both profile files
    2. Identifies major alleles in both samples
    3. Merges profiles based on genomic position
    4. Calculates site presence and allele matching
    5. Tracks statistics for significantly variable sites
    
    Args:
        row: DataFrame row containing 'profile_path_isolate' and 'profile_path_actual'
        significant_p_sites: Set of (contig, position) tuples significant by p-value
        significant_q_sites: Set of (contig, position) tuples significant by q-value
        
    Returns:
        Tuple of (summary_stats, merged_dataframe):
        - summary_stats: Tuple with 10 metrics about the comparison
        - merged_dataframe: Full position-by-position comparison data
    """
    # Handle default arguments
    if significant_p_sites is None:
        significant_p_sites = set()
    if significant_q_sites is None:
        significant_q_sites = set()

    # Extract file paths from the input row
    isolate_path = row["profile_path_isolate"]
    actual_path = row["profile_path_actual"]

    # Load both profile files
    actual_df = read_profile(actual_path)
    isolate_df = read_profile(isolate_path)
    
    # Sanity check: ensure no duplicate positions in the data
    assert not actual_df.duplicated(["contig", "position"]).any(), "actual_df has duplicated sites"
    assert not isolate_df.duplicated(["contig", "position"]).any(), "isolate_df has duplicated sites"

    # Handle missing or empty files
    note = ""
    if actual_df is None or actual_df.empty:
        note = "Actual file not found or empty"
    elif isolate_df is None or isolate_df.empty:
        note = "Isolate file not found or empty"

    if note:
        # Return NaN values for all metrics if files are missing
        return (note, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan), None

    # Identify major alleles in both datasets
    actual_df["major_act"] = get_major_alleles(actual_df)
    isolate_df["major_iso"] = get_major_alleles(isolate_df)

    # Merge the profiles by genomic position (left join keeps all actual positions)
    merged = pd.merge(
        actual_df[["contig", "position", "major_act"]],
        isolate_df[["contig", "position", "major_iso"]],
        on=["contig", "position"],
        how="left",
    )

    # Determine which positions are present (have coverage) in the isolate
    merged['present'] = merged['major_iso'].apply(
        lambda x: isinstance(x, list) and len(x) > 0
    )
    
    # Check if major alleles match between actual and isolate
    merged["match"] = merged.apply(check_allele_match, axis=1)

    # --- Track Significance of Sites ---
    # Create a MultiIndex for efficient membership testing
    merged_sites = pd.MultiIndex.from_frame(merged[['contig', 'position']])
    
    # Mark sites that are significant by p-value (p < 0.05)
    merged['is_significant_by_p_value'] = merged_sites.isin(significant_p_sites)
    merged['is_present_and_significant_by_p_value'] = merged['present'] & merged['is_significant_by_p_value']
    
    # Mark sites that are significant by q-value (FDR-corrected, q < 0.05)
    merged['is_significant_by_q_value'] = merged_sites.isin(significant_q_sites)
    merged['is_present_and_significant_by_q_value'] = merged['present'] & merged['is_significant_by_q_value']

    # --- Calculate Summary Statistics ---
    n_positions_actual = len(merged)  # Total positions in actual sample
    n_positions_present = merged["present"].sum()  # Positions also present in isolate
    n_major_matches = merged["match"].sum()  # Positions where major alleles match
    
    # Count significant sites in actual data
    n_significant_p = merged['is_significant_by_p_value'].sum()
    n_present_significant_p = merged['is_present_and_significant_by_p_value'].sum()
    
    # Count FDR-significant sites in actual data
    n_significant_q = merged['is_significant_by_q_value'].sum()
    n_present_significant_q = merged['is_present_and_significant_by_q_value'].sum()
    
    # Count matches specifically at significant sites that are present in isolate
    n_major_matches_p_sites = (merged['match'] & merged['is_present_and_significant_by_p_value']).sum()
    n_major_matches_q_sites = (merged['match'] & merged['is_present_and_significant_by_q_value']).sum()

    # Package all statistics into a tuple
    summary_stats = (
        "Success",
        n_positions_actual,
        n_positions_present,
        n_major_matches,
        n_significant_p,
        n_present_significant_p,
        n_significant_q,
        n_present_significant_q,
        n_major_matches_p_sites,
        n_major_matches_q_sites
    )

    return summary_stats, merged

In [3]:
# ============================================================================
# STEP A: Load and Prepare Metadata
# ============================================================================
# This section loads metadata for both isolate sequences and actual metagenomic
# samples, then merges them to create matched pairs for comparison.
# 
# Isolate metadata: Information about cultured bacterial isolates
# Actual metadata: Information about metagenomic sequencing samples
# 
# The merge ensures we're comparing isolates with their corresponding 
# metagenomic samples from the same individual at the same timepoint.
# ============================================================================

logging.info("Loading and merging metadata files...")

# Load isolate metadata
isolate_fPath = "/scratch/gpfs/AMOELLER/Phocaeicola_AlleleFlux_Metadata.txt"
isolate_df = pd.read_csv(isolate_fPath, sep="\t")

# Standardize terminology: "Pre-Treatment" -> "pre", "End" -> "end"
isolate_df["time"] = isolate_df["time"].replace({"Pre-Treatment": "pre", "End": "end"})
isolate_df["group"] = isolate_df["group"].replace({"Control": "control", "Fat": "fat"})
isolate_df.drop(["bam_path"], axis=1, inplace=True)

# Load actual metagenomic sample metadata
actual_fPath = "/scratch/gpfs/AMOELLER/diet_manip/copy_sg4230_scratch/sg4230/popgentoolkit/metadata_md_bam.tsv"
actual_df = pd.read_csv(actual_fPath, sep="\t")
actual_df.drop(["cage", "bam_path", "diet", "day", "Sex"], axis=1, inplace=True)

# Merge isolate and actual dataframes to create paired comparisons
# Matching on: group (control/fat), replicate, time (pre/end), and subjectID
df = isolate_df.merge(
    actual_df,
    on=["group", "replicate", "time", "subjectID"],
    suffixes=("_isolate", "_actual"),
)

2025-11-07 13:19:28 - INFO - Loading and merging metadata files...


In [4]:
# ============================================================================
# STEP B: Load and Filter Statistical Significance Data
# ============================================================================
# This section loads results from statistical tests identifying genomic positions
# that show significant allele frequency changes between experimental conditions.
#
# Two significance thresholds are considered:
# 1. p-value < 0.05: Positions with nominally significant changes
# 2. q-value < 0.05: Positions significant after FDR (False Discovery Rate) correction
#
# These "significant sites" represent positions where the actual metagenomic data
# shows real biological variation. We'll track whether isolates preserve these sites.
# ============================================================================

# Specify which MAG (Metagenome-Assembled Genome) to analyze
# mag_id = "SLG221_DASTool_bins_41"
mag_id = "SLG441_DASTool_bins_SLG441_bin.23_sub"

logging.info("Loading and filtering p-value summary file...")

# Load statistical test results comparing different experimental groups
p_value_summary_path = "/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_mapq20/longitudinal/p_value_summary/pre_end-fat_control/p_value_summary_two_sample_paired_pre_end.tsv"
df_p_value = pd.read_csv(p_value_summary_path, sep="\t", dtype={'mag_id': str, 'contig': str, 'position': 'int64'})

# Filter for only the MAG of interest
df_p_value_filtered = df_p_value[df_p_value['mag_id'] == mag_id]
logging.info(f"Filtered p-value summary to {len(df_p_value_filtered):,} rows for mag_id {mag_id}.")

# --- Create set of sites significant by p-value ---
# These positions show nominally significant allele frequency differences
significant_p_df = df_p_value_filtered[
    (df_p_value_filtered['min_p_value'] < 0.05) &
    (df_p_value_filtered['test_type'] == 'two_sample_paired_tTest')
]
significant_p_sites_set = set(zip(significant_p_df['contig'], significant_p_df['position']))
logging.info(f"Found {len(significant_p_sites_set):,} sites significant by p-value (p < 0.05).")

# --- Create set of sites significant by q-value (FDR-corrected) ---
# These positions remain significant after correcting for multiple testing
# This is a more stringent threshold that reduces false positives
significant_q_df = df_p_value_filtered[
    (df_p_value_filtered['q_value'] < 0.05) &
    (df_p_value_filtered['test_type'] == 'two_sample_paired_tTest')
]
significant_q_sites_set = set(zip(significant_q_df['contig'], significant_q_df['position']))
logging.info(f"Found {len(significant_q_sites_set):,} sites significant by q-value (q < 0.05).")

2025-11-07 13:19:28 - INFO - Loading and filtering p-value summary file...
2025-11-07 13:19:30 - INFO - Filtered p-value summary to 24,720 rows for mag_id SLG441_DASTool_bins_SLG441_bin.23_sub.
2025-11-07 13:19:30 - INFO - Found 592 sites significant by p-value (p < 0.05).
2025-11-07 13:19:30 - INFO - Found 67 sites significant by q-value (q < 0.05).


In [5]:
# ============================================================================
# STEP C: Construct File Paths to Profile Data
# ============================================================================
# Build the full file paths to the allele profile files for both isolates
# and actual metagenomic samples. These profiles contain position-by-position
# allele counts (A, C, G, T) for the specified MAG.
#
# File naming convention: {sample_id}/{sample_id}_{mag_id}_profiled.tsv.gz
# ============================================================================

logging.info("Constructing profile file paths...")


# Base directories for profile filesdf  # Display the dataframe with file paths

base_dir_isolate = "/scratch/gpfs/AMOELLER/diet_manip/isolates/single_timepoint/profiles"

base_dir_actual = "/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_mapq20/longitudinal/profiles"
 # Build full paths for isolate profile files
df["profile_path_isolate"] = df["sample_id_isolate"].apply(
    lambda sample_id: f"{base_dir_isolate}/{sample_id}/{sample_id}_{mag_id}_profiled.tsv.gz" if pd.notna(sample_id) else None
)

# Build full paths for actual metagenomic profile files
df["profile_path_actual"] = df["sample_id_actual"].apply(
    lambda sample_id: f"{base_dir_actual}/{sample_id}/{sample_id}_{mag_id}_profiled.tsv.gz" if pd.notna(sample_id) else None
)
df

2025-11-07 13:19:30 - INFO - Constructing profile file paths...


,sample_id_isolate,subjectID,group,replicate,time,sample_id_actual,profile_path_isolate,profile_path_actual
0,1102_A4_10471829_HT5CLAFX5.fna,542,control,4,end,SLG1102,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_m...
1,1102_E6_10471829_HT5CLAFX5.fna,542,control,4,end,SLG1102,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_m...
2,1104_D1_10471829_HT5CLAFX5.fna,544,fat,4,end,SLG1104,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_m...
3,1104_E1_10471829_HT5CLAFX5.fna,544,fat,4,end,SLG1104,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_m...
4,1104_E6_10471829_HT5CLAFX5.fna,544,fat,4,end,SLG1104,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_m...
5,1104_F1_10471829_HT5CLAFX5.fna,544,fat,4,end,SLG1104,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_m...
6,421_C9_10471829_HT5CLAFX5.fna,542,control,4,pre,SLG421,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_m...
7,SLG1205_F4_10473203_HT53MAFX5.fna,562,control,8,end,SLG1205,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_m...
8,SLG1205_G4_10473203_HT53MAFX5.fna,562,control,8,end,SLG1205,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_m...
9,SLG1207_B7_10473203_HT53MAFX5.fna,564,fat,8,end,SLG1207,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_m...


In [6]:
# ============================================================================
# STEP D: Run Parallel Profile Comparisons
# ============================================================================
# Execute the main comparison analysis using multiprocessing for efficiency.
# Each comparison:
# 1. Loads isolate and actual profile files
# 2. Identifies major alleles at each position
# 3. Merges profiles and checks for allele matches
# 4. Tracks presence of significant sites
# 5. Returns summary statistics and detailed comparison data
#
# Using 20 parallel processes to speed up analysis of many sample pairs.
# ============================================================================

output_dir_merged = "/scratch/gpfs/AMOELLER/sidd/isolate_analysis_mapq20"
os.makedirs(output_dir_merged, exist_ok=True)
logging.info(f"Running comparison... Detailed merged files will be saved to '{output_dir_merged}/'")

# Prepare data for parallel processing
rows_to_process = [row for _, row in df.iterrows()]
num_processes = 20
logging.info(f"Starting parallel comparison for {len(df)} pairs using {num_processes} processes...")

# Use functools.partial to "bake in" the sets of significant sites for each worker process
task_function = partial(compare_profiles, significant_p_sites=significant_p_sites_set, significant_q_sites=significant_q_sites_set)

with Pool(processes=num_processes) as pool:
    parallel_results = list(
        tqdm(
            pool.imap(task_function, rows_to_process),
            total=len(rows_to_process),
            desc="Comparing Profiles",
        )
    )

logging.info("Parallel comparison finished.")


2025-11-07 13:19:30 - INFO - Running comparison... Detailed merged files will be saved to '/scratch/gpfs/AMOELLER/sidd/isolate_analysis_mapq20/'
2025-11-07 13:19:30 - INFO - Starting parallel comparison for 31 pairs using 20 processes...
Comparing Profiles: 100%|██████████| 31/31 [00:48<00:00,  1.56s/it]
2025-11-07 13:20:19 - INFO - Parallel comparison finished.


In [7]:
summary_list = [res[0] for res in parallel_results]
merged_df_list = [res[1] for res in parallel_results]

In [8]:
# ============================================================================
# STEP F: Save Detailed Comparison Results
# ============================================================================
# Write out the detailed position-by-position comparison data to files.
# Three types of output files are generated:
# 1. All merged sites: Complete comparison of all genomic positions
# 2. P-significant sites: Only positions with p < 0.05 in actual data
# 3. Q-significant sites: Only positions with q < 0.05 (FDR-corrected)
#
# This allows for focused analysis of biologically significant positions.
# ============================================================================

output_dir_merged = os.path.join("/scratch/gpfs/AMOELLER/sidd/isolate_analysis_mapq20", mag_id)

# Create subdirectories for organized output
output_dir_all = os.path.join(output_dir_merged, f"{mag_id}_all_merged_sites")
output_dir_p_sig = os.path.join(output_dir_merged, f"{mag_id}_p_significant_sites")
output_dir_q_sig = os.path.join(output_dir_merged, f"{mag_id}_q_significant_sites")
os.makedirs(output_dir_all, exist_ok=True)
os.makedirs(output_dir_p_sig, exist_ok=True)
os.makedirs(output_dir_q_sig, exist_ok=True)

logging.info("Saving detailed merged dataframes...")
for i, merged_df in enumerate(tqdm(merged_df_list, desc="Saving Files")):
    if merged_df is not None:
        row = df.iloc[i]
        act_id = row["sample_id_actual"]
        iso_id = row["sample_id_isolate"]
        base_filename = f"merged_{act_id}_vs_{iso_id}"

        # Save the full merged file to its own subdirectory as a TSV
        output_path = os.path.join(output_dir_all, f"{base_filename}.tsv")
        merged_df.to_csv(output_path, index=False, sep='\t')

        # Save the p-value significant subset as a TSV
        p_sig_df = merged_df[merged_df['is_significant_by_p_value']]
        if not p_sig_df.empty:
            p_sig_output_path = os.path.join(output_dir_p_sig, f"{base_filename}_p_significant.tsv")
            p_sig_df.to_csv(p_sig_output_path, index=False, sep='\t')

        # Save the q-value significant subset as a TSV
        q_sig_df = merged_df[merged_df['is_significant_by_q_value']]
        if not q_sig_df.empty:
            q_sig_output_path = os.path.join(output_dir_q_sig, f"{base_filename}_q_significant.tsv")
            q_sig_df.to_csv(q_sig_output_path, index=False, sep='\t')

logging.info("All merged files saved.")

2025-11-07 13:20:19 - INFO - Saving detailed merged dataframes...
Saving Files: 100%|██████████| 31/31 [02:43<00:00,  5.28s/it]
2025-11-07 13:23:03 - INFO - All merged files saved.


In [9]:
# ============================================================================
# STEP G: Compile Summary Statistics
# ============================================================================
# Convert the list of summary tuples into a structured DataFrame and merge
# it with the original metadata. This creates a comprehensive results table
# with both sample information and comparison metrics.
# ============================================================================

# Create DataFrame from summary statistics with descriptive column names
results_df = pd.DataFrame(
    summary_list,
    columns=[
        "comparison_status",                         # Status of comparison: "Success" or error message
        "n_positions_actual",                        # Total genomic positions in actual metagenomic sample
        "n_positions_present",                       # Positions that are also present (have coverage) in isolate
        "n_major_matches",                           # Positions where major alleles match between actual and isolate
        "n_significant_p_in_actual",                 # Number of p-value significant sites (p < 0.05) in actual data
        "n_significant_p_and_present_in_isolate",    # p-significant sites that are retained (have coverage) in isolate
        "n_significant_q_in_actual",                 # Number of q-value significant sites (q < 0.05, FDR-corrected) in actual data
        "n_significant_q_and_present_in_isolate",    # q-significant sites that are retained (have coverage) in isolate
        "n_major_matches_p_sites",                   # Number of matching major alleles specifically at p-significant sites
        "n_major_matches_q_sites"                    # Number of matching major alleles specifically at q-significant sites
    ],
)

# Join the results with the original metadata to create a complete table
# Each row now contains both sample metadata and comparison statistics
final_df = df.join(results_df)

In [10]:
# Use .div() and .fillna(0) for safe division to prevent NaN/inf results.
final_df["site_overlap_fraction"] = final_df["n_positions_present"].div(final_df["n_positions_actual"]).fillna(0)
final_df["major_allele_match_fraction"] = final_df["n_major_matches"].div(final_df["n_positions_present"]).fillna(0)
final_df["p_value_site_retention_fraction"] = final_df["n_significant_p_and_present_in_isolate"].div(final_df["n_significant_p_in_actual"]).fillna(0)
final_df["q_value_site_retention_fraction"] = final_df["n_significant_q_and_present_in_isolate"].div(final_df["n_significant_q_in_actual"]).fillna(0)
final_df["major_allele_match_fraction_p_sites"] = final_df["n_major_matches_p_sites"].div(final_df["n_significant_p_and_present_in_isolate"]).fillna(0)
final_df["major_allele_match_fraction_q_sites"] = final_df["n_major_matches_q_sites"].div(final_df["n_significant_q_and_present_in_isolate"]).fillna(0)

final_df

,sample_id_isolate,subjectID,group,replicate,time,sample_id_actual,profile_path_isolate,profile_path_actual,comparison_status,n_positions_actual,...,n_significant_q_in_actual,n_significant_q_and_present_in_isolate,n_major_matches_p_sites,n_major_matches_q_sites,site_overlap_fraction,major_allele_match_fraction,p_value_site_retention_fraction,q_value_site_retention_fraction,major_allele_match_fraction_p_sites,major_allele_match_fraction_q_sites
0,1102_A4_10471829_HT5CLAFX5.fna,542,control,4,end,SLG1102,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_m...,Success,2087744,...,67,1,4,1,0.001868,0.996667,0.006768,0.014925,1.000000,1.000000
1,1102_E6_10471829_HT5CLAFX5.fna,542,control,4,end,SLG1102,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_m...,Success,2087744,...,67,0,12,0,0.001668,0.999139,0.020305,0.000000,1.000000,0.000000
2,1104_D1_10471829_HT5CLAFX5.fna,544,fat,4,end,SLG1104,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_m...,Success,2088725,...,67,0,0,0,0.000834,0.999426,0.000000,0.000000,0.000000,0.000000
3,1104_E1_10471829_HT5CLAFX5.fna,544,fat,4,end,SLG1104,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_m...,Success,2088725,...,67,0,4,0,0.001256,0.998475,0.008446,0.000000,0.800000,0.000000
4,1104_E6_10471829_HT5CLAFX5.fna,544,fat,4,end,SLG1104,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_m...,Success,2088725,...,67,0,0,0,0.001282,0.998880,0.000000,0.000000,0.000000,0.000000
5,1104_F1_10471829_HT5CLAFX5.fna,544,fat,4,end,SLG1104,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_m...,Success,2088725,...,67,0,6,0,0.001763,0.997284,0.010135,0.000000,1.000000,0.000000
6,421_C9_10471829_HT5CLAFX5.fna,542,control,4,pre,SLG421,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_m...,Success,2072744,...,66,0,0,0,0.000481,0.996991,0.000000,0.000000,0.000000,0.000000
7,SLG1205_F4_10473203_HT53MAFX5.fna,562,control,8,end,SLG1205,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_m...,Success,2095617,...,67,1,30,1,0.015629,0.999206,0.050676,0.014925,1.000000,1.000000
8,SLG1205_G4_10473203_HT53MAFX5.fna,562,control,8,end,SLG1205,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_m...,Success,2095617,...,67,6,128,5,0.068694,0.998284,0.226351,0.089552,0.955224,0.833333
9,SLG1207_B7_10473203_HT53MAFX5.fna,564,fat,8,end,SLG1207,/scratch/gpfs/AMOELLER/diet_manip/isolates/sin...,/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_m...,Success,2095203,...,67,2,10,2,0.004926,0.998741,0.020270,0.029851,0.833333,1.000000


In [11]:
# Define the columns for which to calculate statistics
metrics_to_agg = [
    "site_overlap_fraction",
    "major_allele_match_fraction",
    "p_value_site_retention_fraction",
    "q_value_site_retention_fraction",
    "major_allele_match_fraction_p_sites",
    "major_allele_match_fraction_q_sites"
]
grouped_summary = final_df.groupby(['time', 'group'])[metrics_to_agg].agg(['mean', 'std'])
grouped_summary

site_overlap_fraction           major_allele_match_fraction  \
                              mean       std                        mean   
time group                                                                 
end  control              0.018171  0.022948                    0.998009   
     fat                  0.060255  0.184094                    0.997815   
pre  control              0.176130  0.354278                    0.997975   
     fat                  0.212826  0.303551                    0.998558   

                       p_value_site_retention_fraction            \
                   std                            mean       std   
time group                                                         
end  control  0.001136                        0.062413  0.072167   
     fat      0.002448                        0.085642  0.242143   
pre  control  0.001020                        0.212521  0.375052   
     fat      0.000589                        0.333233  0.317699   

             q_value_site_retention_fraction            \
                                        mean       std   
time group                                               
end  control                        0.022501  0.028765   
     fat                            0.077612  0.235129   
pre  control                        0.211940  0.400267   
     fat                            0.294776  0.344325   

             major_allele_match_fraction_p_sites            \
                                            mean       std   
time group                                                   
end  control                            0.859550  0.348067   
     fat                                0.455910  0.484533   
pre  control                            0.688117  0.388303   
     fat                                0.971234  0.017055   

             major_allele_match_fraction_q_sites            
                                            mean       std  
time group                                                  
end  control                            0.604167  0.503460  
     fat                                0.182000  0.386028  
pre  control                            0.480645  0.477158  
     fat                                0.979758  0.023900

In [12]:
# Save the grouped summary to a new file
grouped_summary_output_path = os.path.join(output_dir_merged, f"{mag_id}_grouped_summary_statistics.tsv")
grouped_summary.to_csv(grouped_summary_output_path, sep='\t')
logging.info(f"Grouped summary statistics saved to {grouped_summary_output_path}")


2025-11-07 13:23:03 - INFO - Grouped summary statistics saved to /scratch/gpfs/AMOELLER/sidd/isolate_analysis_mapq20/SLG441_DASTool_bins_SLG441_bin.23_sub/SLG441_DASTool_bins_SLG441_bin.23_sub_grouped_summary_statistics.tsv


In [13]:
summary_output_path = os.path.join(output_dir_merged, f"{mag_id}_summary_comparison_results.tsv")
final_df.to_csv(summary_output_path, index=False, sep="\t")

## Validation and Quality Checks

analysis is working correctly.

The following cells perform manual validation of the results by examiningspecific examples from the data. This helps verify that the automated

In [14]:
# ============================================================================
# Validation: Inspect a specific comparison example
# ============================================================================
# Look at the first comparison to see the detailed merged results for
# q-value significant sites. This allows manual verification of the analysis.
# ============================================================================

for i, merged_df in enumerate(merged_df_list):
    row = df.iloc[i]
    # print(df)
    act_id = row["sample_id_actual"]
    iso_id = row["sample_id_isolate"]
    base_filename = f"merged_{act_id}_vs_{iso_id}"
    p_sig_df = merged_df[merged_df['is_significant_by_q_value']]
    if i==0:
        print(base_filename)
        break

p_sig_df

merged_SLG1102_vs_1102_A4_10471829_HT5CLAFX5.fna


,contig,position,major_act,major_iso,present,match,is_significant_by_p_value,is_present_and_significant_by_p_value,is_significant_by_q_value,is_present_and_significant_by_q_value
24287,SLG441_DASTool_bins_SLG441_bin.23_sub.fa_k141_...,660,[C],NaN,False,False,True,False,True,False
133493,SLG441_DASTool_bins_SLG441_bin.23_sub.fa_k141_...,110,[G],NaN,False,False,True,False,True,False
140188,SLG441_DASTool_bins_SLG441_bin.23_sub.fa_k141_...,6806,[A],NaN,False,False,True,False,True,False
158457,SLG441_DASTool_bins_SLG441_bin.23_sub.fa_k141_...,3750,[C],NaN,False,False,True,False,True,False
384938,SLG441_DASTool_bins_SLG441_bin.23_sub.fa_k141_...,17284,[T],NaN,False,False,True,False,True,False
...,...,...,...,...,...,...,...,...,...,...
1921004,SLG441_DASTool_bins_SLG441_bin.23_sub.fa_k141_...,15388,[C],NaN,False,False,True,False,True,False
1927966,SLG441_DASTool_bins_SLG441_bin.23_sub.fa_k141_...,850,[C],NaN,False,False,True,False,True,False
2067119,SLG441_DASTool_bins_SLG441_bin.23_sub.fa_k141_...,67896,[T],NaN,False,False,True,False,True,False
2068083,SLG441_DASTool_bins_SLG441_bin.23_sub.fa_k141_...,564,[A],NaN,False,False,True,False,True,False


In [15]:
# Load the isolate profile for sample 1102_A4 for manual inspection
df_1102_A4_isolate = pd.read_csv("/scratch/gpfs/AMOELLER/diet_manip/isolates/single_timepoint/profiles/1102_A4_10471829_HT5CLAFX5.fna/1102_A4_10471829_HT5CLAFX5.fna_SLG221_DASTool_bins_41_profiled.tsv.gz", sep="\t")

In [16]:
# Load the corresponding actual metagenomic profile for sample SLG1102
df_SLG1102 = pd.read_csv("/scratch/gpfs/AMOELLER/diet_manip/AlleleFlux_mapq20/longitudinal/profiles/SLG1102/SLG1102_SLG221_DASTool_bins_41_profiled.tsv.gz", sep="\t")

In [17]:
contigs_of_interest = ["SLG221_DASTool_bins_41.fa_k141_123387"]
position_of_interest = [14734,14746, 14761, 14785, 14794]

In [18]:
# Check allele counts at the specified positions in the actual metagenomic sample
df_SLG1102[(df_SLG1102['contig'].isin(contigs_of_interest)) & (df_SLG1102['position'].isin(position_of_interest))]

,contig,position,ref_base,total_coverage,A,C,G,T,N,mapq_scores,gene_id
267790,SLG221_DASTool_bins_41.fa_k141_123387,14734,T,8,0,5,0,3,0,"42,42,42,23,23,24,24,23",SLG221_DASTool_bins_41.fa_k141_123387_18
267802,SLG221_DASTool_bins_41.fa_k141_123387,14746,C,6,0,1,5,0,0,"42,23,23,24,24,23",SLG221_DASTool_bins_41.fa_k141_123387_18
267817,SLG221_DASTool_bins_41.fa_k141_123387,14761,G,7,5,0,2,0,0,"23,23,24,24,23,42,42",SLG221_DASTool_bins_41.fa_k141_123387_18
267841,SLG221_DASTool_bins_41.fa_k141_123387,14785,A,7,5,0,2,0,0,"24,23,42,42,42,42,42",SLG221_DASTool_bins_41.fa_k141_123387_18
267850,SLG221_DASTool_bins_41.fa_k141_123387,14794,A,7,4,0,3,0,0,"24,24,23,42,42,42,42",SLG221_DASTool_bins_41.fa_k141_123387_18


In [19]:
df_1102_A4_isolate[(df_1102_A4_isolate['contig'].isin(contigs_of_interest)) & (df_1102_A4_isolate['position'].isin(position_of_interest))]

,contig,position,ref_base,total_coverage,A,C,G,T,N,gene_id
273013,SLG221_DASTool_bins_41.fa_k141_123387,14734,T,17,0,0,0,17,0,SLG221_DASTool_bins_41.fa_k141_123387_18
273025,SLG221_DASTool_bins_41.fa_k141_123387,14746,C,16,0,16,0,0,0,SLG221_DASTool_bins_41.fa_k141_123387_18
273040,SLG221_DASTool_bins_41.fa_k141_123387,14761,G,21,0,0,21,0,0,SLG221_DASTool_bins_41.fa_k141_123387_18
273064,SLG221_DASTool_bins_41.fa_k141_123387,14785,A,34,34,0,0,0,0,SLG221_DASTool_bins_41.fa_k141_123387_18
273073,SLG221_DASTool_bins_41.fa_k141_123387,14794,A,33,33,0,0,0,0,SLG221_DASTool_bins_41.fa_k141_123387_18
